In [2]:
import math
import inspect
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.nn import functional as F

from config.notebook_config import NotebookConfig

config = NotebookConfig()


In [30]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        # regularization
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout
        # flash attention make GPU go brrrrr but support is only in PyTorch >= 2.0
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
        if not self.flash:
            print("WARNING: using slow attention. Flash Attention requires PyTorch >= 2.0")
            # causal mask to ensure that attention is only applied to the left in the input sequence
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                        .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)
        print(B,T,C)

        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)
        print(q.shape, k.shape, v.shape )
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        print(f'{k.shape=}')
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        # causal self-attention; Self-attend: (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
        # if self.flash:
        #     # efficient attention using Flash Attention CUDA kernels
        #     y = torch.nn.functional.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0, is_causal=True)
        #     print(y.shape)
        # else:
            # manual implementation of attention
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))

        bias = torch.tril(torch.ones(config.block_size, config.block_size)).view(1, 1, config.block_size, config.block_size)

        # att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = att.masked_fill(bias[:,:,:T,:T] == 0, float('-inf'))

        att = F.softmax(att, dim=-1) 
        att = self.attn_dropout(att)
        y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side

        # output projection
        y = self.resid_dropout(self.c_proj(y))
        return y


In [32]:
data = torch.randn((config.batch_size, config.sequence_length, config.n_embd))  # batch_size, sequence length, embedding dimension

attention = CausalSelfAttention(config)
attention(data).shape


10 50 256
torch.Size([10, 50, 256]) torch.Size([10, 50, 256]) torch.Size([10, 50, 256])
k.shape=torch.Size([10, 8, 50, 32])


torch.Size([10, 50, 256])

In [10]:
bias = torch.tril(torch.ones(config.block_size, config.block_size)).view(1, 1, config.block_size, config.block_size)
bias.shape

torch.Size([1, 1, 128, 128])

In [27]:
a = torch.tensor([3,40,8,1,100], dtype = float) / 50
a = F.softmax(a, dim = 0)
a = torch.round(a, decimals = 3)
a


tensor([0.0830, 0.1730, 0.0910, 0.0790, 0.5740], dtype=torch.float64)

In [8]:
from model import GPT
from config.notebook_config import NotebookConfig as cfg
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
idx = torch.randint(0,cfg.vocab_size,(cfg.batch_size,cfg.block_size))
idx.shape

torch.Size([10, 1024])

In [7]:
import importlib
import model
model = importlib.reload(model)
from model import GPT

In [9]:
model = GPT(cfg)
x,logits,loss = model.forward(idx)
logits.shape

number of parameters: 16.04M
cpu
shape of output from all layers = torch.Size([10, 1024, 256])


torch.Size([10, 1, 50304])

In [9]:
from datasets import load_dataset

ds = load_dataset("DhruvDancingBuddha/osho_discourses")

In [10]:
ds

DatasetDict({
    train: Dataset({
        features: ['char_url', 'topic_name', 'topic_lesson_name', 'topic_lesson_url', 'all_txt'],
        num_rows: 1965
    })
})

In [11]:
text1 = ds['train'][0]['all_txt']
words = text1.split()
len(words)

8754

In [ ]:
# Wrap continuous text: newline every WORDS_PER_LINE words,
# blank line (paragraph) every LINES_PER_PARA lines.
WORDS_PER_LINE = 12   # use 10-15
LINES_PER_PARA = 5

words = text1.split()
lines = [
    " ".join(words[i:i + WORDS_PER_LINE])
    for i in range(0, len(words), WORDS_PER_LINE)
]

paragraphs = [
    "\n".join(lines[i:i + LINES_PER_PARA])
    for i in range(0, len(lines), LINES_PER_PARA)
]

formatted_text = "\n\n".join(paragraphs)

with open("output.txt", "w", encoding="utf-8") as f:
    f.write(formatted_text)

print(f"Wrote {len(words)} words -> {len(lines)} lines -> {len(paragraphs)} paragraphs to output.txt")


In [15]:
import nltk

nltk.download("punkt")

from nltk.tokenize import sent_tokenize

# with open("input.txt", "r", encoding="utf-8") as f:
#     text = f.read()

sentences = sent_tokenize(text1)

# text_new = "\n".join(sentences)
# print(text_new)
# with open("output.txt", "w", encoding="utf-8") as f:
#     f.write("\n".join(sentences))

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [18]:
sentences[0]

'When Rabbi Birnham lay dying,his wife burst into tears.He said, “What are you crying for?My whole life was only that I might learn how to die.”Life is in living.'

In [7]:
import torch
import torch.nn.functional as F

In [37]:
logits = torch.randn((1,1,512))*10
temperature = 1.2
# # pluck the logits at the final step and scale by desired temperature
logits = logits[:, -1, :] / temperature
# # optionally crop the logits to only the top k options
# if top_k is not None:
#     v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
#     logits[logits < v[:, [-1]]] = -float('Inf')
# # apply softmax to convert logits to (normalized) probabilities
probs = F.softmax(logits, dim=-1) #.sort(descending = False)
top_k_values, top_k_index = torch.topk(probs,10,sorted= True)

# # sample from the distribution
sampled_position = torch.multinomial(top_k_values, num_samples=1)
idx_next = top_k_index.gather(-1, sampled_position)
# # append sampled index to the running sequence and continue
# idx = torch.cat((idx, idx_next), dim=1)

In [38]:
idx_next

tensor([[253]])

In [20]:
x = torch.tensor([1, 5, 2, 4, 3])

values, indices = torch.topk(x, 2)

# values:  tensor([5, 4])
values
# indices: tensor([1, 3])

tensor([5, 4])

In [40]:
top_k_values.sum()

tensor(0.9906)